# Setup - Course Environment

This notebook sets up the Spark session and downloads the NYC Taxi data used throughout the course modules.

## Initialize Spark Session

First, let's start our Spark session. This may take a while if starting for the first time as dependencies are downloaded.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("DataModeling") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

Spark 4.0.1 initialized!


## Create Namespace

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")
print("Namespace 'taxi' created!")

Namespace 'taxi' created!


## Download NYC Taxi Data

We'll download NYC Yellow Taxi trip data from **June through October 2023** (5 months). This dataset contains:
- Pick-up and drop-off times and locations
- Trip distances
- Fare amounts
- Payment types
- Passenger counts

**Note**: We're downloading 5 months of data (~225MB total). This will give us enough data to see meaningful differences between partitioning strategies.

In [3]:
import urllib.request
import os
import boto3
from botocore.client import Config

# Configure MinIO client using S3-compatible API
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download NYC Yellow Taxi data for June through October 2023 and upload to MinIO
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
minio_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

# Create temp directory
os.makedirs(temp_dir, exist_ok=True)

# Download data for months 6-10 (June through October)
months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    minio_key = f"{minio_prefix}/{filename}"
    
    try:
        # Check if file already exists locally
        if os.path.exists(local_path):
            print(f"{filename} already exists locally, skipping download")
        else:
            # Download file
            print(f"Downloading {filename} (~45MB)...")
            urllib.request.urlretrieve(url, local_path)
            print(f"  Downloaded to {local_path}")
        
        # Upload to MinIO
        print(f"  Uploading to MinIO: s3a://{bucket_name}/{minio_key}...")
        s3_client.upload_file(local_path, bucket_name, minio_key)
        print(f"  Uploaded successfully!")
        uploaded_files.append(minio_key)
        
        # Clean up local file
        os.remove(local_path)
    except Exception as e:
        print(f"  Error processing {filename}: {e}")

print(f"\nAll files ready in MinIO! Total: {len(uploaded_files)} months of data")
print(f"Location: s3a://{bucket_name}/{minio_prefix}/")

  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-06.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-06.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-07.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-07.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-08.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-08.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-09.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-09.parquet...
  Uploaded successfully!
  Downloaded to /tmp/nyc_taxi_download/yellow_tripdata_2023-10.parquet
  Uploading to MinIO: s3a://warehouse/raw/yellow_tripdata_2023-10.parquet...
  Uploaded successfully!

All files ready in MinIO! Total: 5 months of data
Location: s3a://warehouse/raw/


## Verify Data

Let's verify the data was downloaded and uploaded correctly by reading it with Spark.

In [4]:
# Read all parquet files from MinIO
s3_path = "s3a://warehouse/raw/"

taxi_df = spark.read.parquet(s3_path)

print(f"Reading from: {s3_path}")
print(f"Total records: {taxi_df.count():,}")
print(f"\nSchema:")
taxi_df.printSchema()

Reading from: s3a://warehouse/raw/
Total records: 15,407,558

Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



## Setup Complete!

The environment is now ready for the course modules. You can proceed to:
- Module 1: Apache Iceberg Fundamentals
- Module 2: Taking Advantage of Iceberg Tables
- Module 3: Operating and Optimizing Apache Iceberg